In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
import requests

repo = Path(r'c:/Users/Tushar/Downloads/aiaic-ml/aiaic-ml-foundation')
os.chdir(repo)
for line in (repo / '.env').read_text(encoding='utf-8').splitlines():
    if line and '=' in line and not line.startswith('#'):
        k, v = line.split('=', 1)
        os.environ[k.strip()] = v.strip()

src_file = repo / 'out' / 'agmarknet_mh_mp_2016_2026.csv.gz'
dst = repo / 'data' / 'raw' / 'agmarknet_mh_mp_2016_2026.csv.gz'
dst.parent.mkdir(parents=True, exist_ok=True)
if src_file.exists() and not dst.exists():
    shutil.copy2(src_file, dst)
    print('COPIED', dst)
print('BASE', os.getenv('AIAIC_API_BASE'))
base = os.getenv('AIAIC_API_BASE', os.getenv('VITE_AQIAIC_BASE_URL', 'https://disarm-scrubbed-pushiness.ngrok-free.dev')).rstrip('/')
for path in ['/health', '/catalog?state=maharashtra']:
    r = requests.get(base + path, timeout=20)
    print(path, r.status_code, r.text[:200].replace('\n', ' '))

cmd = [
    sys.executable, '-m', 'src.forecast.train',
    '--input', 'data/raw/agmarknet_mh_mp_2016_2026.csv.gz',
    '--crops', 'Onion', 'Soyabean', 'Wheat',
    '--horizons', '7', '30',
    '--holdout-days', '180',
    '--step-days', '7',
    '--seed', '42',
    '--models-dir', 'models/forecast_official',
    '--source', 'Agmarknet via data.gov.in (Variety-wise Daily Market Prices), AIAIC export 2016-2026'
]
print('RUN', cmd)
p = subprocess.run(cmd, cwd=repo, capture_output=True, text=True)
print(p.stdout)
print(p.stderr)
if p.returncode != 0:
    raise SystemExit(p.returncode)

jsons = sorted((repo / 'models' / 'forecast_official').glob('forecast_*.json'))
print('TARGET JSON', jsons[-1])
cmp = subprocess.run([sys.executable, 'compare_with_official_backtest.py', str(jsons[-1])], cwd=repo / 'aryan files', capture_output=True, text=True)
print(cmp.stdout)
print(cmp.stderr)
if cmp.returncode != 0:
    raise SystemExit(cmp.returncode)

test_cmd = [sys.executable, '-m', 'pytest', 'tests/forecast', '-q']
p2 = subprocess.run(test_cmd, cwd=repo, capture_output=True, text=True)
print(p2.stdout)
print(p2.stderr)
if p2.returncode != 0:
    raise SystemExit(p2.returncode)